# FinSight 360: Notebook 01 - Data Understanding & Schema Inspection

**Author:** FinSight 360 Analytics Team  
**Objective:** Profile raw lending data, evaluate schema completeness, inspect missingness patterns, and verify target variable distributions.


In [ ]:
import pandas as pd
import numpy as np
from src.config import load_config
from src.ingest import ingest_raw_data

config = load_config()
raw_df, meta = ingest_raw_data(config['paths']['raw_data'])
print('Raw Records:', f"{meta['total_rows_ingested']:,}")
print('Columns Count:', meta['total_cols_ingested'])
print('Memory Footprint:', f"{meta['memory_mb']:.2f} MB")
raw_df.head(3)

## 1. Data Dictionary & Column Definitions
Inspect the official metadata definitions provided by LendingClub.

In [ ]:
info_df = pd.read_csv(config['paths']['data_dictionary'])
pd.set_option('display.max_colwidth', None)
info_df.head(10)

## 2. Missing Value Profiling
Identify missing fields in the raw source extract and compute missingness proportions.

In [ ]:
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct.round(2)})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values(by='missing_count', ascending=False)
print('Columns with Missing Values:')
missing_df

## 3. Terminal Target Outcome Distribution (`loan_status`)
Verify that all facilities represent completed loan contracts (`Fully Paid` vs `Charged Off`).

In [ ]:
status_dist = raw_df['loan_status'].value_counts(dropna=False)
status_pct = raw_df['loan_status'].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({'count': status_dist, 'share_pct': status_pct.round(2)})
print('Loan Status Distribution:')
print(target_summary)
print(f"\nPortfolio Baseline Default Rate: {status_pct['Charged Off']:.2f}%")

## 4. Key Underwriting Numerical Features Distribution
Inspect summary statistics across key underwriting numerical variables.

In [ ]:
num_cols = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti']
raw_df[num_cols].describe().round(2).T

## Key Takeaways from Data Understanding
1. **Volume & Granularity**: 396,030 completed facilities spanning $5.59B in funded capital.
2. **Clean Closed Statuses**: No active or in-flight loans; 80.39% Fully Paid and 19.61% Charged Off.
3. **Missingness Isolated**: Missing values are confined to non-core underwriting fields (e.g., `mort_acc`, `emp_title`) and tracked in the Data Quality audit framework.